<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# Load the dataset
df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

Dataset shape: (30000, 44)
Columns: 44


In [ ]:
feature_cols = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
    "age_days"
]

print("Model features:")
for col in feature_cols:
    print("-", col)

print("\nNumber of features:", len(feature_cols))

Model features:
- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share
- age_days

Number of features: 6


In [ ]:
march_data = con.sql(f"""
WITH previous_window AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS previous_30d_impressions,
        SUM(f.gsc_clicks) AS previous_30d_clicks,
        AVG(f.gsc_avg_position) AS previous_30d_avg_position
    FROM {TABLES['fact_daily']} f
    WHERE f.report_date >= DATE '2026-02-01'
      AND f.report_date < DATE '2026-03-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY
        f.client_hash_id,
        f.content_hash_id
),

march_window AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']} f
    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY
        f.client_hash_id,
        f.content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_query_count,
        MAX(impressions_90d)
            / NULLIF(SUM(impressions_90d), 0) AS top_query_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
),

content_age AS (
    SELECT
        content_hash_id,
        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '2026-03-01'
        ) AS age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
)

SELECT
    p.client_hash_id,
    p.content_hash_id,

    p.previous_30d_impressions,
    p.previous_30d_clicks,
    p.previous_30d_avg_position,

    q.visible_query_count,
    q.top_query_share,

    a.age_days,

    m.march_impressions

FROM previous_window p

LEFT JOIN march_window m
    ON p.client_hash_id = m.client_hash_id
   AND p.content_hash_id = m.content_hash_id

LEFT JOIN query_signals q
    ON p.content_hash_id = q.content_hash_id

LEFT JOIN content_age a
    ON p.content_hash_id = a.content_hash_id

WHERE p.previous_30d_impressions > 0
  AND m.march_impressions IS NOT NULL
""").df()

print("March modeling rows:", len(march_data))
march_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March modeling rows: 134238


,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share,age_days,march_impressions
0,client_08a6a72ff48e62c0,content_95991e493be7daea,3332.0,4.0,7.356512,28,0.126202,163,3218.0
1,client_08a6a72ff48e62c0,content_959fc7aa6b4aadc2,10.0,0.0,27.566667,3,0.761468,320,72.0
2,client_08a6a72ff48e62c0,content_95ba6de2bc2a794c,50.0,0.0,18.459615,3,0.520000,320,95.0
3,client_08a6a72ff48e62c0,content_95be7a732080b5cc,1309.0,1.0,21.177232,12,0.172269,214,813.0
4,client_08a6a72ff48e62c0,content_95c314ab78eabcf4,3.0,0.0,6.250000,2,0.617647,209,40.0


In [ ]:
march_data["is_declining"] = (
    march_data["march_impressions"]
    < 0.80 * march_data["previous_30d_impressions"]
).astype(int)

print("Target distribution:")
print(march_data["is_declining"].value_counts())

print("\nTarget rate:")
print(round(march_data["is_declining"].mean(), 4))

Target distribution:
is_declining
0    107413
1     26825
Name: count, dtype: int64

Target rate:
0.1998


In [ ]:
X = march_data[feature_cols].copy()
y = march_data["is_declining"].copy()

groups = march_data["client_hash_id"].copy()

# Handle missing numeric values consistently
X = X.replace([np.inf, -np.inf], np.nan)

for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test base rate:", round(y_test.mean(), 4))

Training rows: 107390
Test rows: 26848
Test base rate: 0.1998


In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("Average Precision:",
      round(average_precision_score(y_test, test_prob), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_prob), 4))

Average Precision: 0.4016
ROC-AUC: 0.7404


In [ ]:
queue = march_data.loc[
    X_test.index,
    ["client_hash_id", "content_hash_id"] + feature_cols
].copy()

queue["predicted_decline_probability"] = test_prob

# Rank highest-risk pages first
queue = queue.sort_values(
    "predicted_decline_probability",
    ascending=False
).reset_index(drop=True)

# Add priority
queue["priority"] = pd.cut(
    queue["predicted_decline_probability"],
    bins=[-np.inf, 0.30, 0.60, np.inf],
    labels=["Monitor", "Review", "High Priority"]
)

print("Ranked queue:")
display(queue.head(10))

Ranked queue:


,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share,age_days,predicted_decline_probability,priority
0,client_3ffa76342f366962,content_80a56002a30036d6,4.0,0.0,0.000000,<NA>,NaN,192,1.00,High Priority
1,client_a80fca3f171ed1de,content_30ca7d7ffa2c3aad,225.0,0.0,12.038852,<NA>,NaN,17,1.00,High Priority
2,client_3ffa76342f366962,content_6f4420b9c2aad1c3,2.0,0.0,7.500000,<NA>,NaN,183,1.00,High Priority
3,client_3ffa76342f366962,content_71e46fd2b35e7d51,4.0,0.0,0.000000,<NA>,NaN,209,1.00,High Priority
4,client_a80fca3f171ed1de,content_194ab331fda5e6a4,229.0,0.0,11.002292,<NA>,NaN,17,1.00,High Priority
5,client_3ffa76342f366962,content_e8cf370129639017,4.0,0.0,0.000000,<NA>,NaN,192,1.00,High Priority
6,client_65de48885f4ef01b,content_2879353f031b1983,3.0,0.0,5.666667,<NA>,NaN,202,1.00,High Priority
7,client_a80fca3f171ed1de,content_89be0b2b0f2f8e3e,193.0,0.0,11.263961,<NA>,NaN,17,1.00,High Priority
8,client_a80fca3f171ed1de,content_9396155fd65f8b39,163.0,0.0,11.607312,<NA>,NaN,17,0.99,High Priority
9,client_a80fca3f171ed1de,content_5c75f6b73df38f9f,195.0,0.0,10.953489,<NA>,NaN,17,0.99,High Priority


In [ ]:
def get_reason_codes(row):
    reasons = []

    # Safely check numeric values
    impressions = row["previous_30d_impressions"]
    clicks = row["previous_30d_clicks"]
    avg_position = row["previous_30d_avg_position"]
    query_count = row["visible_query_count"]
    age_days = row["age_days"]

    if pd.notna(impressions) and impressions < 100:
        reasons.append("Low recent impressions")

    if pd.notna(clicks) and clicks < 5:
        reasons.append("Low recent clicks")

    if pd.notna(query_count) and query_count < 5:
        reasons.append("Low query visibility")

    if pd.notna(avg_position) and avg_position > 20:
        reasons.append("Poor average position")

    if pd.notna(age_days) and age_days >= 365:
        reasons.append("Older content")

    # Fallback if no specific reason was triggered
    if not reasons:
        reasons.append("Model-prioritized review")

    return "; ".join(reasons)

In [ ]:
queue["reason_code"] = queue.apply(get_reason_codes, axis=1)

display(
    queue[
        [
            "content_hash_id",
            "predicted_decline_probability",
            "priority",
            "reason_code"
        ]
    ].head(20)
)

,content_hash_id,predicted_decline_probability,priority,reason_code
0,content_80a56002a30036d6,1.00,High Priority,Low recent impressions; Low recent clicks
1,content_30ca7d7ffa2c3aad,1.00,High Priority,Low recent clicks
2,content_6f4420b9c2aad1c3,1.00,High Priority,Low recent impressions; Low recent clicks
3,content_71e46fd2b35e7d51,1.00,High Priority,Low recent impressions; Low recent clicks
4,content_194ab331fda5e6a4,1.00,High Priority,Low recent clicks
5,content_e8cf370129639017,1.00,High Priority,Low recent impressions; Low recent clicks
6,content_2879353f031b1983,1.00,High Priority,Low recent impressions; Low recent clicks
7,content_89be0b2b0f2f8e3e,1.00,High Priority,Low recent clicks
8,content_9396155fd65f8b39,0.99,High Priority,Low recent clicks
9,content_5c75f6b73df38f9f,0.99,High Priority,Low recent clicks


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use

This playbook is intended to support SEO specialists and content managers in prioritizing webpages for human review. The ranking provides a directional decision-support signal based on observed historical search-performance features.

The score should be used to decide which pages to investigate first, not as an automatic instruction to update a page. A high score does not establish that a refresh will improve rankings, traffic, or conversions.

The recommendations are most useful when the underlying search-performance data is recent and comparable with the data used to train the model. Changes in search behavior, content strategy, or measurement systems may reduce the usefulness of the ranking over time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human review rules

Before taking action, a content specialist should review the page's current purpose, search intent, content quality, recent changes, and business relevance. The model score should be treated as a prioritization signal rather than a final decision.

No-go cases

The system should not automatically:

publish or modify webpage content;
delete or redirect pages;
change titles or metadata without review;
claim that a page will gain traffic or rankings;
make decisions based only on the model score;
override expert judgment when the page has important business or editorial context.

Human review remains necessary because the model does not observe every factor that affects whether a content refresh is appropriate.

In [ ]:
print("Human-review queue size:", len(queue))

print("\nPriority distribution:")
print(queue["priority"].value_counts())

print("\nTop reason codes:")
print(
    queue["reason_code"]
    .value_counts()
    .head(10)
)

Human-review queue size: 26848

Priority distribution:
priority
Monitor          20550
Review            4962
High Priority     1336
Name: count, dtype: int64

Top reason codes:
reason_code
Low recent impressions; Low recent clicks                                                 6769
Low recent clicks                                                                         5650
Model-prioritized review                                                                  3396
Low recent clicks; Low query visibility                                                   2862
Low recent impressions; Low recent clicks; Low query visibility                           1720
Low recent impressions; Low recent clicks; Poor average position                          1427
Low recent clicks; Poor average position                                                  1125
Low recent impressions; Low recent clicks; Low query visibility; Poor average position     681
Low recent clicks; Older content                  

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring and retraining

The recommendations should be reviewed periodically rather than assumed to remain valid indefinitely.

Retraining or investigation should be considered if:

the distribution of model scores changes substantially;
the positive/declining base rate changes materially;
feature distributions shift from the training data;
validation Average Precision falls materially;
new content types or measurement definitions are introduced;
human reviewers consistently disagree with the model's highest-priority recommendations.

A model should not be retrained simply because a newer model is available. Retraining should be tied to evidence that the current model's decision-support value has degraded.

In [ ]:
print("Monitoring snapshot")
print("-------------------")

print("Rows in queue:", len(queue))
print(
    "Mean predicted decline probability:",
    round(queue["predicted_decline_probability"].mean(), 4)
)

print(
    "Median predicted decline probability:",
    round(queue["predicted_decline_probability"].median(), 4)
)

print("\nPriority distribution:")
print(queue["priority"].value_counts())

Monitoring snapshot
-------------------
Rows in queue: 26848
Mean predicted decline probability: 0.1984
Median predicted decline probability: 0.14

Priority distribution:
priority
Monitor          20550
Review            4962
High Priority     1336
Name: count, dtype: int64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/content_action_queue.csv"

queue.to_csv(output_path, index=False)

print("Exported:", output_path)
print("Rows exported:", len(queue))

Exported: work/outputs/content_action_queue.csv
Rows exported: 26848


In [ ]:
import json

metrics = {
    "model": "RandomForestClassifier",
    "average_precision": round(
        average_precision_score(y_test, test_prob), 4
    ),
    "roc_auc": round(
        roc_auc_score(y_test, test_prob), 4
    ),
    "test_rows": int(len(y_test)),
    "test_base_rate": round(float(y_test.mean()), 4),
    "feature_count": len(feature_cols)
}

metrics_path = "work/outputs/w07_action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics:", metrics_path)

Saved metrics: work/outputs/w07_action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.